In [1]:
# ============================================================
# WEEK 12 BBO NOTEBOOK
# PCA-Inspired Optimisation Strategy
# Week 1–11 inputs and outputs embedded directly
# ============================================================

import numpy as np
import warnings

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
np.random.seed(42)

# ============================================================
# WEEK 1–11 INPUT DATA
# ============================================================

inputs_by_week = [
[
np.array([0.211325,0.788675]), np.array([0.723607,0.276393]), np.array([0.166667,0.5,0.833333]), np.array([0.125,0.375,0.625,0.875]), np.array([0.875,0.625,0.375,0.125]), np.array([0.15,0.35,0.55,0.75,0.95]), np.array([0.9,0.1,0.7,0.3,0.5,0.8]), np.array([0.111111,0.222222,0.333333,0.444444,0.555555,0.666667,0.777778,0.888889])
],
[
np.array([0.654299,0.353479]), np.array([0.754299,0.253479]), np.array([0.304299,0.553479,0.709691]), np.array([0.804299,0.603479,0.409691,0.205016]), np.array([0.854299,0.653479,0.359691,0.155016]), np.array([0.904299,0.703479,0.509691,0.305016,0.103275]), np.array([0.884299,0.123479,0.729691,0.285016,0.523275,0.787492]), np.array([0.104299,0.243479,0.319691,0.465016,0.573275,0.647492,0.794889,0.860861])
],
[
np.array([0.582812,0.421077]), np.array([0.712865,0.284413]), np.array([0.118496,0.481282,0.876608]), np.array([1.0,0.683447,0.334333,0.0]), np.array([0.882245,0.615032,0.380358,0.114494]), np.array([0.0,0.226282,0.564108,0.905744,1.0]), np.array([0.905495,0.091782,0.689608,0.305244,0.491854,0.804378]), np.array([0.113495,0.214782,0.338108,0.437244,0.549353,0.673378,0.771789,0.898699])
],
[
np.array([0.183704,0.127166]), np.array([0.255353,0.749841]), np.array([0.094906,0.770362,0.859589]), np.array([0.820856,0.226975,0.784625,0.113609]), np.array([0.91786,0.273128,0.475575,0.052446]), np.array([0.052509,0.789636,0.354234,0.228337,0.889991]), np.array([0.833796,0.096195,0.232612,0.789158,0.185769,0.681185]), np.array([0.448564,0.335191,0.751694,0.277261,0.167614,0.717472,0.304117,0.767059])
],
[
np.array([0.582812,0.421077]), np.array([0.684812,0.281383]), np.array([0.171685,0.508728,0.819757]), np.array([0.734042,0.687625,0.466427,0.303264]), np.array([0.568707,0.909615,0.957143,0.01142]), np.array([0.744968,0.084644,0.801527,0.169728,0.990511]), np.array([1.0,0.229371,0.690054,0.244152,0.521424,1.0]), np.array([0.02262,0.0,0.288709,0.762817,0.551656,0.961269,0.843087,1.0])
],
[
np.array([0.582812,0.421077]), np.array([0.996744,0.999561]), np.array([0.166667,0.5,0.833333]), np.array([0.743137,0.718436,0.58181,0.365984]), np.array([0.572707,0.914615,0.952143,0.01642]), np.array([0.236209,0.379873,0.584786,0.843391,1.0]), np.array([0.024461,0.059677,0.99533,0.299568,0.038576,0.691027]), np.array([0.148189,0.189483,0.270757,0.392507,0.446351,0.670593,0.665417,1.0])
],
[
np.array([0.582812,0.421077]), np.array([0.744767,0.322712]), np.array([0.167431,0.503246,0.833129]), np.array([0.692844,0.522126,0.400511,0.337375]), np.array([0.543363,0.994173,0.995,0.003824]), np.array([0.236209,0.379873,0.584786,0.843391,1.0]), np.array([0.194327,0.008117,0.940397,0.331469,0.120428,0.708672]), np.array([0.208946,0.148993,0.202639,0.316908,0.431312,0.503474,0.634079,0.995])
],
[
np.array([0.582812,0.421077]), np.array([0.723607,0.276393]), np.array([0.17358,0.509985,0.861627]), np.array([0.605953,0.443908,0.321881,0.510101]), np.array([0.503541,0.995,0.995,0.000001]), np.array([0.339419,0.513193,0.599128,0.825009,0.789136]), np.array([0.232309,0.000001,0.881882,0.376082,0.191527,0.731904]), np.array([0.04585,0.409304,0.057579,0.179582,0.499647,0.251108,0.757574,0.939082])
],
[
np.array([0.582812,0.421077]), np.array([0.69434,0.207302]), np.array([0.147117,0.526235,0.849011]), np.array([0.524137,0.403433,0.22003,0.482874]), np.array([0.577252,0.995,0.995,0.000001]), np.array([0.482662,0.445135,0.505979,0.901513,0.790255]), np.array([0.234062,0.000001,0.862390,0.432759,0.245568,0.773410]), np.array([0.246667,0.211931,0.156108,0.312021,0.424946,0.497662,0.627877,0.995])
],
[
np.array([0.582812,0.421077]), np.array([0.718442,0.086744]), np.array([0.155756,0.525144,0.938413]), np.array([0.658829,0.135745,0.01475,0.885838]), np.array([0.614625,0.995,0.995,0.000001]), np.array([0.396866,0.534494,0.843367,0.854385,0.577951]), np.array([0.236193,0.000001,0.838386,0.411651,0.294881,0.727030]), np.array([0.246667,0.211931,0.156108,0.312021,0.424946,0.497662,0.627877,0.995])
],
[
np.array([0.582812,0.421077]), np.array([0.723607,0.276393]), np.array([0.14903,0.507335,0.866594]), np.array([0.574526,0.455077,0.168845,0.479246]), np.array([0.614625,0.995,0.995,0.000001]), np.array([0.396866,0.534494,0.843367,0.854385,0.577951]), np.array([0.236193,0.000001,0.838386,0.411651,0.294881,0.727030]), np.array([0.246667,0.211931,0.156108,0.312021,0.424946,0.497662,0.627877,0.995])
]
]

# ============================================================
# WEEK 1–11 OUTPUT DATA
# ============================================================

outputs_by_week = [
np.array([1.1327056288856873e-125,0.5675786892822564,-0.032385408076210126,-17.20744048260943,60.223125,-1.3287857969718009,0.34356041660314957,9.0501517903694]),
np.array([1.1867858443665185e-41,0.2715258567299176,-0.1198581070659559,-13.082213230390916,50.179390256321376,-1.5080002951000964,0.31639485635485903,9.0219949134074]),
np.array([7.99676981308551e-19,0.5213723465552891,-0.04726977098841539,-25.67625344929702,64.78245026474816,-1.7372122723701597,0.3507828450585503,9.0587238074074]),
np.array([-2.410121917336285e-100,0.0244939290195959,-0.04207093453964322,-20.330739763644917,61.278397876784794,-2.4624737429462145,0.0634803557047261,8.275283250689899]),
np.array([7.99676981308551e-19,0.4258127251524913,-0.06232295859499999,-12.496845976106417,942.2521944777988,-2.280900502246122,0.21828066675598462,8.427686115001]),
np.array([7.99676981308551e-19,-0.004122974640885927,-0.032385408076210126,-14.192389833871584,943.6841142765069,-1.2257941580841207,0.4410410448155631,9.3346877420455]),
np.array([7.99676981308551e-19,0.3799297936947829,-0.03240117791304375,-6.92709780967855,1723.425126888365,-1.2923495282910902,0.919847131115406,9.472143824862]),
np.array([7.99676981308551e-19,0.5381696913344642,-0.02792717356049581,-4.634066674167709,1679.193994434957,-1.0832489053663208,1.3511467710792968,9.1699591109441]),
np.array([7.99676981308551e-19,0.5123530018001279,-0.025522227618805376,-4.5906554634204895,1784.4636825839586,-1.153199093198391,1.3736252613501883,9.472748274068]),
np.array([7.99676981308551e-19,0.5380835457242255,-0.17435713522547233,-23.112309779047532,1857.0401615205615,-0.9469472223006096,1.6718981508700566,9.472748274068]),
np.array([7.99676981308551e-19,0.42635833571459203,-0.027698163149178882,-6.704147291575101,1857.0401615205615,-0.8973667566949034,1.6718981508700566,9.472748274068])
]

# ============================================================
# PCA-BASED OPTIMISATION FUNCTIONS
# ============================================================

def get_history(function_index):
    X = np.vstack([week[function_index] for week in inputs_by_week])
    y = np.array([week[function_index] for week in outputs_by_week])
    return X, y


def choose_pca_components(X):
    max_components = min(X.shape[0] - 1, X.shape[1])
    if max_components < 1:
        return 1
    return max_components


def fit_pca_model(X):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    n_components = choose_pca_components(X)

    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

    # keep enough PCs to retain at least 90% variance
    keep = np.searchsorted(cumulative_variance, 0.90) + 1
    keep = max(1, min(keep, n_components))

    X_pca_reduced = X_pca[:, :keep]

    return scaler, pca, X_pca_reduced, keep, cumulative_variance


def cluster_in_pca_space(X_pca, y):
    k = min(3, len(X_pca))

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = kmeans.fit_predict(X_pca)

    cluster_scores = {}

    for c in range(k):
        idx = labels == c
        cluster_scores[c] = {
            "mean_output": float(np.mean(y[idx])),
            "max_output": float(np.max(y[idx])),
            "centroid": np.mean(X_pca[idx], axis=0)
        }

    best_cluster = max(cluster_scores, key=lambda c: cluster_scores[c]["mean_output"])

    return labels, cluster_scores, best_cluster


def train_surrogate(X_pca, y):
    model = RandomForestRegressor(
        n_estimators=700,
        max_depth=5,
        random_state=42
    )
    model.fit(X_pca, y)
    return model


def inverse_pca_point(z, scaler, pca, keep, original_dim):
    full_z = np.zeros(pca.n_components_)
    full_z[:keep] = z

    x_scaled = pca.inverse_transform(full_z.reshape(1, -1))
    x_original = scaler.inverse_transform(x_scaled)

    return np.clip(x_original.ravel(), 0.000001, 0.995000)


def generate_pca_candidates(X, y, function_index, n_candidates=20000):
    scaler, pca, X_pca, keep, cum_var = fit_pca_model(X)

    labels, cluster_scores, best_cluster = cluster_in_pca_space(X_pca, y)

    model = train_surrogate(X_pca, y)

    best_idx = np.argmax(y)
    best_z = X_pca[best_idx]
    best_cluster_center = cluster_scores[best_cluster]["centroid"]

    dim = X.shape[1]

    # Function-specific PCA search behaviour
    if function_index == 4:
        # Function 5: very strong, exploit tightly in reduced PCA space
        radius = 0.08
        global_ratio = 0.03

    elif function_index in [2, 3, 5]:
        # Functions 3, 4, 6: still weak/negative, wider PCA exploration
        radius = 0.35
        global_ratio = 0.25

    elif function_index in [6, 7]:
        # Functions 7 and 8: positive, moderate PCA refinement
        radius = 0.18
        global_ratio = 0.10

    else:
        radius = 0.22
        global_ratio = 0.15

    n_global = int(n_candidates * global_ratio)
    n_local = n_candidates - n_global

    # Local PCA candidates around best point and best cluster
    local_around_best = best_z + np.random.normal(0, radius, size=(n_local // 2, keep))
    local_around_cluster = best_cluster_center + np.random.normal(0, radius * 1.25, size=(n_local // 2, keep))

    # Global PCA candidates inside observed PCA bounds
    mins = X_pca.min(axis=0)
    maxs = X_pca.max(axis=0)

    global_candidates = np.random.uniform(
        mins,
        maxs,
        size=(n_global, keep)
    )

    z_candidates = np.vstack([
        local_around_best,
        local_around_cluster,
        global_candidates,
        X_pca
    ])

    preds = model.predict(z_candidates)

    # Prefer high predicted output but penalise excessive PCA distance
    distance = np.linalg.norm(z_candidates - best_z, axis=1)

    if function_index == 4:
        score = preds - 0.25 * distance
    elif function_index in [2, 3, 5]:
        score = preds - 0.04 * distance
    else:
        score = preds - 0.08 * distance

    best_candidate_z = z_candidates[np.argmax(score)]

    x_candidate = inverse_pca_point(
        best_candidate_z,
        scaler,
        pca,
        keep,
        dim
    )

    return (
        np.round(x_candidate, 6),
        keep,
        cum_var,
        cluster_scores,
        best_cluster,
        np.max(y),
        X[np.argmax(y)]
    )


# ============================================================
# GENERATE WEEK 12 INPUTS
# ============================================================

week12_inputs = []

print("WEEK 12 PCA-BASED BBO INPUT GENERATION")
print("======================================\n")

for function_index in range(8):

    X, y = get_history(function_index)

    candidate, keep, cum_var, cluster_scores, best_cluster, best_y, best_x = generate_pca_candidates(
        X,
        y,
        function_index
    )

    week12_inputs.append(candidate)

    print(f"Function {function_index + 1}")
    print(f"Best historical output: {best_y}")
    print(f"Best historical input:  {np.round(best_x, 6)}")
    print(f"PCA components kept:    {keep}")
    print(f"Variance retained:      {cum_var[keep-1]:.4f}")
    print(f"Best PCA cluster:       {best_cluster}")
    print(f"Week 12 suggested input:{candidate}")
    print()

print("PORTAL-READY WEEK 12 INPUTS")
print("===========================\n")

for i, arr in enumerate(week12_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

WEEK 12 PCA-BASED BBO INPUT GENERATION

Function 1
Best historical output: 7.99676981308551e-19
Best historical input:  [0.582812 0.421077]
PCA components kept:    2
Variance retained:      1.0000
Best PCA cluster:       0
Week 12 suggested input:[0.582812 0.421077]

Function 2
Best historical output: 0.5675786892822564
Best historical input:  [0.723607 0.276393]
PCA components kept:    2
Variance retained:      1.0000
Best PCA cluster:       0
Week 12 suggested input:[0.721744 0.274122]

Function 3
Best historical output: -0.025522227618805376
Best historical input:  [0.147117 0.526235 0.849011]
PCA components kept:    2
Variance retained:      0.9482
Best PCA cluster:       1
Week 12 suggested input:[0.154633 0.530187 0.856486]

Function 4
Best historical output: -4.5906554634204895
Best historical input:  [0.524137 0.403433 0.22003  0.482874]
PCA components kept:    3
Variance retained:      0.9773
Best PCA cluster:       1
Week 12 suggested input:[0.541769 0.384974 0.253223 0.63257